# 4 · Map the next observation with the same model and preparation

Notebook 3 saved the trained model as a release. A release holds the architecture and weights; it does not say which four bands the model saw, whether reflectance was decoded, or how missing pixels were filled. Those choices live in your preparation code, and changing them by accident changes every map.

We reload the release in a fresh kernel, write the preparation down as one function, and prove that together they reproduce notebook 3's map. Then we keep an image/prediction pair for review.

**Before starting:** finish notebooks [2](02_sources.ipynb) and [3](03_inputs.ipynb), including notebook 3's **Keep this trained model** cell. Start a fresh kernel with `examples/` as the working directory. This lesson uses the saved files; it does not download imagery or train again.

**At the end:** `data/landcover/review/` holds an image and its land-cover prediction as one catalogued Item. Rerunning replaces these tutorial files.

In [ ]:
from pathlib import Path

import torch
import xarray as xr
import yaml
from transformers import AutoModel

import geosave as gs
import geosave.model.release  # registers the GeoSave model type with Transformers

folder = Path("data/landcover")
model = AutoModel.from_pretrained(folder, local_files_only=True).eval()
classes = yaml.safe_load((folder / "classes.yaml").read_text())

## Write down the preparation once

These are notebook 3's choices: blue, green, red and near-infrared in that order, decoded reflectance, zero where nothing was observed. The observation mask is returned too: filling an input must not turn a missing pixel into a reported class.

In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]


def prepare_image(scene):
    reflectance = scene[BANDS].gs.mask_and_scale()
    observed = reflectance.to_array().notnull().all("variable")
    return reflectance.fillna(0), observed

`map_land_cover` is notebook 3's prediction loop for one scene: overlapping chips, blended scores, then classes.

In [ ]:
def map_land_cover(image, observed):
    tree = xr.DataTree.from_dict({"image": image})
    chips = gs.Chipper([tree], size=128, stride=64)
    merger = gs.ChipMerger(chips.layout, window="hamming")

    with torch.inference_mode():
        for index in range(len(chips)):
            pixels = chips[index]["image"].to_dataset().gs.to_tensor(dtype="float32")
            scores = model(image=pixels.unsqueeze(0))[0].numpy()
            merger.add(index, scores)

    cover = merger.merge(0).argmax("band").astype("uint8")
    cover = cover.where(observed, classes["ignore_index"]).rename("land_cover")
    cover = cover.gs.write_nodata(classes["ignore_index"])
    return cover.gs.write_flags(classes["class_map"], classes["color_map"])

## Check that the release reproduces notebook 3's map

Map the most recent forest observation and compare it with the map notebook 3 saved for the same date. Both catalogs name Items by date.

In [ ]:
forest = gs.open_vector("data/khdtk/catalog.parquet")
latest = forest.sort_values("datetime").iloc[[-1]]
scene = latest.gs.to_dataset().squeeze("time").load()

image, observed = prepare_image(scene)
cover = map_land_cover(image, observed)

saved = gs.open_vector("data/khdtk_pred/catalog.parquet")
saved = saved[saved["id"] == latest["id"].item()]
previous = saved.gs.to_dataset().squeeze("time")["label"]

agreement = (cover.values == previous.values).mean()
f"{agreement:.1%} of pixels match notebook 3"

In [ ]:
scene.gs.plot(rgb=("B04", "B03", "B02"), title="Forest observation") + cover.gs.plot(
    title="Reloaded model, same preparation"
)

## Keep the image and prediction together

A reviewer needs to see the source image beside the model's claim. Put both in one Dataset and publish it as one dated Item, as notebook 2 did with image/label pairs.

In [ ]:
review = scene[BANDS].assign(land_cover=cover).expand_dims("time")

review_catalog = review.gs.to_catalog(
    folder / "review", id_format="{start:%Y%m%d}", overwrite=True
)
review_catalog.gs.to_geoparquet(folder / "review/catalog.parquet", overwrite=True)
review_catalog[["id", "datetime"]]

## More things you can do with this model (optional)

The review pair completes this lesson. These are possible next steps for your project.

### Keep a small reference case with every model version
Retain one observation and its expected map. Check them before mapping a new season or after upgrading dependencies. That catches an accidental band reorder or second reflectance scaling before it reaches a whole region.

### Reuse the preparation when building a new training set
Call `prepare_image` before constructing the training Dataset, as notebook 3 did. Keep label preparation separate: class IDs must not receive reflectance scaling. Keep paired augmentation in the training loop.

### Hand the model to a colleague
Share `data/landcover/` with `prepare_image`. `GeoSaveModel.push_to_hub` publishes the release to Hugging Face; the preparation still travels as your code.

### Test a pretrained encoder fairly
A pretrained model may require different bands or normalization. Make that a new experiment with its own preparation function and the same geographic splits, not a silent change to this model's inputs.